# Modul 1 — Pengantar Citra Digital: Representasi, Karakteristik, dan Aplikasi dalam Visi Komputer

**Praktikum Pengolahan Citra dan Visi Komputer — Minggu 1**
Jurusan Teknologi Informasi, Politeknik Negeri Malang

Notebook ini adalah hasil **rework jobsheet Minggu 1 dari Google Colab ke Kaggle Notebook**.
Seluruh isi jobsheet dikerjakan di sini: ulasan teori, praktikum D1, praktikum D2,
jawaban pertanyaan, dan tugas praktikum.

| Bagian | Isi |
|---|---|
| C. Ulasan Teori | citra digital & visi komputer, tools, NumPy/ndarray, OpenCV, format citra & model warna, resolusi |
| D1 | Pengenalan GitHub & Google Colab (narasi langkah, tanpa kode) |
| D2 | Praktikum: import library, baca & tampilkan citra (cv2 / skimage / PIL / matplotlib), resize setengah, konversi BGR<->RGB, gabung citra asli + hasil konversi, ukuran citra, akses & ubah pixel |
| Pertanyaan D2 | 5 pertanyaan dijawab pada sel markdown |
| Tugas D2 | 4 tugas dengan keluaran visual |

> **Catatan lingkungan.** Dataset berada di `/kaggle/input/pcvk-images/week1/` ketika dijalankan di
> Kaggle dan otomatis memakai `datasets/week1/` ketika dijalankan lokal (lihat sel konfigurasi di
> bawah). Tidak ada `google.colab`, `drive.mount`, maupun `files.upload()`; fungsi `cv2_imshow`
> disediakan oleh shim berbasis matplotlib karena Kaggle tidak punya GUI. Citra minggu ini:
> `fruits.jpg`, `butterfly.jpg`, `starry_night.jpg`.

In [ ]:
# ==== KONFIGURASI LINGKUNGAN (jalan di Kaggle maupun lokal) ====
import os, sys, glob, math
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

def _find_base():
    for p in ('/kaggle/input/pcvk-images',
              '/kaggle/input/datasets/singgihwahyupermana/pcvk-images'):
        if os.path.isdir(p):
            return p
    _hit = glob.glob('/kaggle/input/**/pcvk-images', recursive=True)
    if _hit:
        return _hit[0]
    d = os.getcwd()
    for _ in range(6):
        if all(os.path.isdir(os.path.join(d, f'week{i}')) for i in (1, 2, 3)):
            return d
        d = os.path.dirname(d)
    return None

BASE = _find_base()
WEEK = os.path.join(BASE, 'week1')   # <-- ganti angka minggu sesuai notebook

def show(img, title='', cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

# Shim cv2_imshow (Colab) -> Kaggle
try:
    from google.colab.patches import cv2_imshow
except Exception:
    def cv2_imshow(img):
        show(img)

print('BASE :', BASE)
print('WEEK :', WEEK)
print('files:', sorted(os.listdir(WEEK)) if WEEK and os.path.isdir(WEEK) else 'MISSING')

## A. Tujuan Praktikum

1. Mampu membuat repository di GitHub dan membuat/menyimpan Python Notebook (Colab/Kaggle).
2. Mampu membuka file citra dari kode Python.
3. Mampu mengakses dan mengubah pixel pada citra.
4. Mampu memahami dasar-dasar library OpenCV di Python.
5. Mampu memahami channel warna pada OpenCV serta konversinya.

**Alat dan bahan:** PC/laptop, akun GitHub, akun Kaggle (pengganti Google Colab), serta citra
`fruits.jpg`, `butterfly.jpg`, dan `starry_night.jpg` yang sudah tersedia pada dataset minggu ini.

---

## C. Ulasan Teori

### C1. Pengantar citra digital dan visi komputer

**Citra digital** adalah representasi numerik dari citra dua dimensi. Nilai numeriknya disimpan pada
elemen citra yang disebut **pixel** (elemen gambar terkecil), umumnya berupa bilangan biner 8 bit
(0–255) yang tersusun sebagai *map raster*, yaitu array dua dimensi bertipe integer dengan jumlah
baris dan kolom yang tetap. Karakteristik citra digital ditentukan oleh:

* **Resolusi** — jumlah pixel horizontal x vertikal (mis. 1920x1080 untuk Full HD).
* **Bit depth** — banyaknya bit per pixel, menentukan jumlah warna yang dapat direpresentasikan
  (8 bit = 256 level per kanal).
* **Format penyimpanan** — menentukan kompresi dan fitur tambahan (mis. transparansi).

**Pengolahan Citra Digital** mempelajari algoritma transformasi citra (mis. perbaikan kontras,
filtering, segmentasi), sedangkan **visi komputer** adalah cabang AI yang membuat mesin mampu
"melihat dan memahami" citra: klasifikasi, deteksi objek, pelacakan, sampai segmentasi universal
(SAM 2) dan Vision Transformer. Paradigma modern bergeser dari *handcrafted features* (HOG, SIFT)
menuju *end-to-end deep learning* yang dibantu GPU berdaya tinggi dan dataset skala besar.

**Contoh aplikasi (2025):** kesehatan (analisis X-ray, rekonstruksi 3D organ), industri (inspeksi
kualitas produk), pertanian & perikanan (deteksi penyakit tanaman, penghitung benih ikan),
transportasi (kendaraan otonom), keamanan (pengenalan wajah), dan media (AR/VR).

### C2. Tools pendukung

| Tool | Peran pada praktikum ini |
|---|---|
| **Python** | bahasa utama; sintaks sederhana, open source, ekosistem pustaka pengolahan citra sangat lengkap |
| **GitHub** | version control & kolaborasi, tempat menyimpan notebook dan dataset/model terbuka |
| **Google Colab / Kaggle** | notebook berbasis cloud, tanpa instalasi, menyediakan GPU/TPU gratis |
| **IDE desktop** (VS Code, PyCharm, Jupyter/Anaconda, Spyder) | alternatif pengembangan lokal |

Pada jobsheet asli, notebook dijalankan di Google Colab. Versi rework ini dijalankan di **Kaggle
Notebook**: sama-sama notebook Jupyter berbasis cloud, tetapi dataset sudah otomatis ter-mount di
`/kaggle/input` sehingga tidak perlu mengunggah file atau me-mount Google Drive.

### C1.1 NumPy dan objek `ndarray`

NumPy (Numerical Python) menyediakan objek **N-dimensional array (`ndarray`)**. Dibandingkan `list`
Python, `ndarray` lebih hemat memori dan jauh lebih cepat karena operasinya di-vektorisasi
(diimplementasikan di C). Pada dimensi NumPy, **axis** adalah sumbu array dan banyaknya axis disebut
**rank**. Contoh: `[[1., 1., 1.], [2., 2., 2.]]` adalah array ber-rank 2 dengan axis-0 panjang 2 dan
axis-1 panjang 3.

| Atribut | Kegunaan |
|---|---|
| `ndarray.ndim` | jumlah dimensi array (rank) |
| `ndarray.shape` | ukuran array di setiap dimensi, matriks (n, m) |
| `ndarray.size` | jumlah elemen array = perkalian elemen `shape` |
| `ndarray.dtype` | tipe data elemen array |
| `ndarray.itemsize` | ukuran setiap elemen dalam byte (float64 -> 8 byte = 64/8) |
| `ndarray.data` | buffer berisi elemen array |
| `ndarray.nbytes` | total byte seluruh elemen (`size` x `itemsize`) |

Metode utama untuk memanipulasi data: **indexing** (`a[5]`), **slicing** (`a[2:6]`, `a[:5]`,
`a[::2]`), **iterating** (`for row in b:` dan `for element in b.flat:`), serta **reshaping**
(`a.reshape(6, 2)`).

In [ ]:
# C1.1 - Atribut ndarray
a = np.array([3., 6., 1., 5., 3., 2., 0., 0., 2., 5., 1., 4.])
print('a           :', a)
print('a.ndim      :', a.ndim)
print('a.shape     :', a.shape)
print('a.size      :', a.size)
print('a.dtype     :', a.dtype)
print('a.itemsize  :', a.itemsize, 'byte')
print('a.nbytes    :', a.nbytes, 'byte')

b = a.reshape(6, 2)          # reshaping: 12 elemen -> 6 baris x 2 kolom
print('\nb = a.reshape(6, 2):')
print(b)
print('b.ndim =', b.ndim, '| b.shape =', b.shape, '| b.size =', b.size)

c = np.array([[1., 1., 1.], [2., 2., 2.]])
print('\nc (rank 2) shape =', c.shape, '-> axis-0 =', c.shape[0], ', axis-1 =', c.shape[1])

d = np.arange(12, dtype=np.uint8).reshape(3, 4)
print('\nd dtype =', d.dtype, '| itemsize =', d.itemsize, '| nbytes =', d.nbytes)

In [ ]:
# C1.1 - Indexing, slicing, iterating, reshaping
x = np.array([10, 20, 30, 40, 50, 60, 70, 80])
print('x            :', x)
print('x[5]         :', x[5])        # indexing
print('x[2:6]       :', x[2:6])      # slicing index 2 s.d. 5
print('x[:5]        :', x[:5])       # 5 elemen pertama
print('x[::2]       :', x[::2])      # jarak 2 elemen

m = np.arange(1, 13).reshape(3, 4)
print('\nm =')
print(m)
print('m[1, 2]      :', m[1, 2])     # satu elemen
print('m[:, 1]      :', m[:, 1])     # satu kolom
print('m[1, :]      :', m[1, :])     # satu baris
print('m[0:2, 1:3]  :')
print(m[0:2, 1:3])                 # subarray

print('iterating per baris:')
for row in m:
    print('   ', row)

print('iterating per elemen (m.flat):')
for element in m.flat:
    print('   ', element)

print('reshape(-1)   :', m.reshape(-1))   # kembali ke 1 dimensi
print('transpose m.T :'); print(m.T)

### C1.2 OpenCV (Open Source Computer Vision Library)

OpenCV dikembangkan Intel Corporation dan merupakan library open source untuk pengolahan citra
digital serta banyak metode AI. Versi pertamanya diperkenalkan Gary Bradsky (1999) dan dirilis tahun
2000; kini bersifat *standalone* dan multiplatform (Windows, Linux, macOS, Android) dengan binding
Python, Java, dan MATLAB.

| Versi | Tahun | Fitur utama | Fokus |
|---|---|---|---|
| 1.x | 2000–2006 | fungsi dasar citra, Haar Cascade | pengolahan citra dasar (C/C++) |
| 2.x | 2009–2015 | API modern C++, modul `ml`, CUDA awal | peralihan ke C++ & akselerasi GPU |
| 3.x | 2015–2018 | modularisasi (core, imgproc, video, ml), contrib (SIFT/SURF), tracking | computer vision klasik + tracking |
| 4.x | 2018–2023 | modul `dnn` (TensorFlow/PyTorch/Caffe/ONNX), OpenVINO/TensorRT | integrasi deep learning |
| 5.x | 2024–2025 | optimasi GPU, dukungan ViT/YOLOv9/SAM 2, WebAssembly | jembatan CV klasik <-> AI modern |

Lima library dasar OpenCV: **CV** (algoritma image processing & computer vision), **ML**
(machine learning), **Highgui** (GUI, image & video I/O), **CXCORE** (struktur data, XML), dan
**CvAux** (khusus computer vision). Fitur utamanya meliputi image & video I/O, pemrosesan citra low
dan mid level, modul computer vision high level, metode AI/ML, sampling & transformasi citra,
analisis citra biner, pemodelan 3D, deteksi gerak, analisis struktural, kalibrasi kamera, dan
pelabelan citra.

**Hal paling penting untuk diingat:** OpenCV membaca citra berwarna dalam urutan kanal **BGR**
(bukan RGB). Konsekuensinya, library yang mengasumsikan RGB — seperti matplotlib, PIL, dan skimage —
akan menampilkan warna tertukar bila diberi array hasil `cv.imread` tanpa konversi.

In [ ]:
# C1.2 - Verifikasi instalasi & kemampuan dasar OpenCV
print('Versi OpenCV :', cv.__version__)
print('Versi NumPy  :', np.__version__)
print()
print('Ketersediaan fungsi dasar OpenCV:')
for name in ['imread', 'imwrite', 'resize', 'cvtColor', 'split', 'merge', 'hconcat',
             'vconcat', 'line', 'rectangle', 'filter2D', 'Canny', 'getStructuringElement']:
    print('   cv.{:<22} : {}'.format(name, hasattr(cv, name)))

fruits = os.path.join(WEEK, 'fruits.jpg')

# Membaca citra: berwarna (default) vs grayscale
img_color = cv.imread(fruits, cv.IMREAD_COLOR)        # hasil: BGR, 3 kanal
img_gray  = cv.imread(fruits, cv.IMREAD_GRAYSCALE)    # hasil: 1 kanal
print('\ncv.IMREAD_COLOR     -> shape', img_color.shape, '| dtype', img_color.dtype)
print('cv.IMREAD_GRAYSCALE -> shape', img_gray.shape,  '| dtype', img_gray.dtype)
print('pixel [0,0] berwarna (B, G, R):', img_color[0, 0])
print('pixel [0,0] grayscale         :', img_gray[0, 0])

---

## D. PRAKTIKUM

### D1. Pengenalan GitHub dan Google Colab

Bagian ini bersifat prosedural (tidak menghasilkan kode). Langkah aslinya di Colab, dan padanannya
saat memakai Kaggle Notebook ditulis dalam tanda kurung.

1. **Membuat akun GitHub** di <https://github.com/join> — isi seluruh field wajib, klik *complete
   setup*, lalu verifikasi email.
2. **Membuat repository baru** melalui tombol *Create a Repository*: isi nama repo
   (mis. `PCVK_Ganjil_2026`), deskripsi opsional, pilih *public*/*private*, lalu klik
   *Create repository*.
3. Repository siap diisi project perkuliahan PCVK Ganjil 26/27.
4. **Membuka Google Colaboratory** di <https://colab.research.google.com/>.
   *(Padanan Kaggle: menu **Code → New Notebook** di <https://www.kaggle.com/code>.)*
5. **Menghubungkan Colab dengan GitHub**: pilih menu *GitHub*, masukkan URL akun
   (`https://github.com/nama_akun`), centang *Sertakan repositori pribadi* bila repo bersifat
   private, lalu *Authorize googlecolab*. Notebook baru dibuat dengan tombol **New notebook**.
   *(Padanan Kaggle: `File → Import Notebook` dari GitHub, atau unggah file `.ipynb`.)*
6. **Menyimpan notebook ke GitHub**: ganti nama file `*.ipynb`, lalu
   `File → Save a copy in GitHub`, pilih repository `PCVK_Ganjil_2026`, klik **OK**.
7. Notebook terbuka di halaman GitHub; tombol **Open in Colab** memverifikasi bahwa file berhasil
   diakses Colab (ikon GitHub muncul di kiri atas). Notebook berisi sel **teks terformat** (`+ Text`)
   dan sel **kode Python** (`+ Code`) yang bisa langsung dijalankan.

### D2. Pengenalan Python untuk membaca dan menampilkan image

Praktikum D2 memakai beberapa library: **NumPy, pandas, OpenCV (cv2), scikit-image, PIL, dan
matplotlib**. Langkah-langkahnya adalah: import library -> baca citra -> resize menjadi setengah ->
konversi warna ke RGB -> gabungkan citra asli dengan citra hasil konversi -> tampilkan -> cek ukuran
citra -> akses dan ubah pixel.

#### Langkah 1 — Import library

In [ ]:
# Langkah 1 - import library yang dipakai pada praktikum ini
import numpy as np
import pandas as pd
import cv2 as cv
from skimage import io
from PIL import Image
import matplotlib.pylab as pylab      # pada jobsheet Colab: import matplotlib.pylab as plt

# Pada Colab, cv2_imshow diimpor dari google.colab.patches.
# Di Kaggle tidak ada google.colab, sehingga dipakai shim cv2_imshow dari sel konfigurasi.
print('numpy     ', np.__version__)
print('pandas    ', pd.__version__)
print('cv2       ', cv.__version__)
import skimage; print('skimage   ', skimage.__version__)
import PIL;     print('PIL       ', PIL.__version__)
import matplotlib; print('matplotlib', matplotlib.__version__)
print('\ncv2_imshow ->', cv2_imshow)

#### Langkah 2 — Membaca dan menampilkan image

Pada jobsheet asli, citra diambil dari URL GitHub OpenCV. Karena dataset sudah tersedia pada
lingkungan ini, daftar URL tetap ditulis sebagai dokumentasi, lalu setiap URL dipetakan ke berkas
lokal yang sesuai di `WEEK`. Untuk setiap citra dilakukan lima operasi: **baca citra -> resize
setengah -> konversi BGR ke RGB -> gabungkan citra asli dengan citra hasil konversi -> tampilkan**.

In [ ]:
# Langkah 2 - daftar URL citra (sesuai jobsheet) dan padanannya pada dataset lokal
imageurls = [
    'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/fruits.jpg',
    'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/butterfly.jpg',
    'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/starry_night.jpg',
]
imagenames = ['fruits.jpg', 'butterfly.jpg', 'starry_night.jpg']
imgpaths   = [os.path.join(WEEK, n) for n in imagenames]

for url, name, path in zip(imageurls, imagenames, imgpaths):
    print('URL :', url)
    print('  ->', path, '| tersedia:', os.path.exists(path))

In [ ]:
# Langkah 2 - baca, resize setengah, konversi BGR->RGB, gabungkan, tampilkan
for name, path in zip(imagenames, imgpaths):
    img       = cv.imread(path)                                  # dibaca OpenCV -> BGR
    img_half  = cv.resize(img, (0, 0), fx=0.5, fy=0.5)           # resize jadi setengah
    img_rgb   = cv.cvtColor(img_half, cv.COLOR_BGR2RGB)          # konversi BGR -> RGB
    img_merge = cv.hconcat([img_half, img_rgb])                  # gabung citra asli + hasil konversi

    print('{}: asli {} -> setengah {} | gabungan {}'.format(
        name, img.shape, img_half.shape, img_merge.shape))
    print('  pixel [0,0] asli (B,G,R) :', img_half[0, 0], '| setelah konversi (R,G,B):', img_rgb[0, 0])

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].imshow(cv.cvtColor(img_half, cv.COLOR_BGR2RGB))
    ax[0].set_title('{} - dibaca sebagai BGR (benar)'.format(name)); ax[0].axis('off')
    ax[1].imshow(img_rgb)
    ax[1].set_title('{} - hasil konversi ke RGB (benar)'.format(name)); ax[1].axis('off')
    plt.tight_layout(); plt.show()

    # cv2_imshow (shim) menafsirkan array sebagai BGR: separuh kiri benar, separuh kanan tertukar
    print('  cv2_imshow(img_merge) - separuh kiri BGR asli, separuh kanan RGB hasil konversi:')
    cv2_imshow(img_merge)

**Pengamatan Langkah 2.** Dua panel di atas tampak identik karena keduanya sudah ditampilkan dengan
urutan kanal yang benar (BGR dikonversi dulu, RGB langsung). Perbedaannya baru terlihat pada
`cv2_imshow(img_merge)`: shim/`cv2_imshow` menganggap seluruh array berurutan **BGR**, sehingga
separuh kiri (citra BGR asli) tampil benar, sedangkan separuh kanan (hasil konversi yang sudah RGB)
justru tertukar kanal merah dan birunya. Inilah bukti visual bahwa `cv.imread` menghasilkan BGR dan
`cv.cvtColor(..., cv.COLOR_BGR2RGB)` memang mengubah urutan kanal, bukan sekadar menyalin data.

**Pembacaan citra dengan library lain.** Selain OpenCV, citra yang sama dapat dibaca memakai
`skimage.io.imread` dan `PIL.Image.open`. Keduanya menghasilkan array **RGB** (urutan kanal berbeda
dari OpenCV), sehingga hasilnya bisa langsung ditampilkan matplotlib tanpa konversi.

In [ ]:
# Langkah 2 - perbandingan cara membaca citra: OpenCV, skimage, PIL, matplotlib
path = imgpaths[0]                       # fruits.jpg
img_cv   = cv.imread(path)                                  # BGR
img_sk   = io.imread(path)                                  # RGB
img_pil  = np.asarray(Image.open(path))                     # RGB
img_mpl  = plt.imread(path)                                 # RGB (uint8 untuk JPEG, float32 untuk PNG)

print('cv.imread      :', img_cv.shape,  img_cv.dtype,  '| pixel [0,0]:', img_cv[0, 0])
print('io.imread      :', img_sk.shape,  img_sk.dtype,  '| pixel [0,0]:', img_sk[0, 0])
print('Image.open     :', img_pil.shape, img_pil.dtype, '| pixel [0,0]:', img_pil[0, 0])
print('plt.imread     :', img_mpl.shape, img_mpl.dtype, '| pixel [0,0]:', img_mpl[0, 0])
print('ukuran identik :', img_cv.shape == img_sk.shape == img_pil.shape == img_mpl.shape)
print('cv.imread BGR == kebalikan kanal RGB dari skimage?',
      np.array_equal(img_cv, img_sk[:, :, ::-1]))

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(img_sk);                                     ax[0].set_title('skimage.io.imread (RGB)'); ax[0].axis('off')
ax[1].imshow(img_pil);                                    ax[1].set_title('PIL.Image.open (RGB)');   ax[1].axis('off')
ax[2].imshow(cv.cvtColor(img_cv, cv.COLOR_BGR2RGB));      ax[2].set_title('cv.imread + BGR->RGB');   ax[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Langkah 2 - konversi dua arah BGR <-> RGB dan pemisahan kanal
img      = cv.imread(imgpaths[0])                 # BGR
img_rgb  = cv.cvtColor(img, cv.COLOR_BGR2RGB)     # BGR -> RGB
img_back = cv.cvtColor(img_rgb, cv.COLOR_RGB2BGR) # RGB -> BGR (kembali)

print('konversi bolak-balik menghasilkan citra identik:', np.array_equal(img, img_back))
print('pixel [100,100] BGR:', img[100, 100], '-> RGB:', img_rgb[100, 100], '-> BGR lagi:', img_back[100, 100])

b, g, r = cv.split(img)                 # memisahkan kanal (hasil OpenCV: B, G, R)
print('\nshape tiap kanal:', b.shape, g.shape, r.shape)
print('rata-rata intensitas kanal -> B: {:.1f} | G: {:.1f} | R: {:.1f}'.format(
    b.mean(), g.mean(), r.mean()))

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB)); ax[0].set_title('citra asli (RGB)'); ax[0].axis('off')
ax[1].imshow(b, cmap='gray'); ax[1].set_title('kanal B'); ax[1].axis('off')
ax[2].imshow(g, cmap='gray'); ax[2].set_title('kanal G'); ax[2].axis('off')
ax[3].imshow(r, cmap='gray'); ax[3].set_title('kanal R'); ax[3].axis('off')
plt.tight_layout(); plt.show()

#### Langkah 3 — Melihat ukuran file image

Ukuran citra dapat dibaca dari atribut `ndarray`: `shape` (tinggi, lebar, jumlah kanal), `size`
(jumlah elemen = tinggi x lebar x kanal), `dtype` dan `itemsize` (tipe & ukuran elemen), serta
`nbytes` (ukuran data di memori). Bandingkan pula dengan ukuran berkas di disk, karena berkas JPEG
sudah dikompresi sehingga jauh lebih kecil daripada data mentah di memori.

In [ ]:
# Langkah 3 - ukuran citra: shape, size, dtype, itemsize, nbytes, ukuran berkas
for name, path in zip(imagenames, imgpaths):
    img = cv.imread(path)
    h, w, ch = img.shape
    print('=== {} ==='.format(name))
    print('  shape            : {}  -> tinggi {} px, lebar {} px, kanal {}'.format(img.shape, h, w, ch))
    print('  size (elemen)    : {}  = {} x {} x {}'.format(img.size, h, w, ch))
    print('  dtype / itemsize : {} / {} byte'.format(img.dtype, img.itemsize))
    print('  nbytes (memori)  : {} byte = {:.2f} MB'.format(img.nbytes, img.nbytes / 1024**2))
    print('  ukuran berkas    : {} byte = {:.2f} KB (terkompresi JPEG)'.format(
        os.path.getsize(path), os.path.getsize(path) / 1024))
    print('  resolusi         : {} x {} = {} pixel'.format(w, h, w * h))

print('\nPerbandingan RGB vs grayscale (teori: 3:1)')
img  = cv.imread(imgpaths[0])
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
print('  RGB      : shape {} | nbytes {} byte'.format(img.shape, img.nbytes))
print('  grayscale: shape {} | nbytes {} byte'.format(gray.shape, gray.nbytes))
print('  rasio    : {:.2f} : 1'.format(img.nbytes / gray.nbytes))

#### Langkah 4 — Mengakses pixel: menggambar garis horizontal hijau di tengah citra

Pixel diakses dengan **indexing** `img[baris, kolom]` dan diubah dengan penugasan
`img[baris, kolom] = [B, G, R]`. Karena urutan kanal OpenCV adalah BGR, warna hijau murni ditulis
`[0, 255, 0]`. Untuk membuat garis horizontal di tengah citra, seluruh kolom pada baris tengah
(`h // 2`) diisi nilai tersebut.

In [ ]:
# Langkah 4 - akses pixel: garis horizontal hijau di tengah citra
img = cv.imread(imgpaths[0])            # fruits.jpg, shape (480, 512, 3)
img_line = img.copy()

print('bentuk citra   :', img_line.shape)
print('pixel [0,0]    :', img_line[0, 0], '(B, G, R)')
print('pixel [10,20]  :', img_line[10, 20])
print('baris terakhir :', img_line[-1, -1])

h, w = img_line.shape[:2]
row = h // 2                                    # baris tengah
print('\nbaris tengah =', row, '| lebar citra =', w, 'pixel')

for col in range(w):                            # akses pixel satu per satu
    img_line[row, col] = [0, 255, 0]            # BGR hijau murni

print('pixel sebelum :', img[row, 0])
print('pixel sesudah :', img_line[row, 0])
print('jumlah pixel pada garis:', int(np.sum(np.all(img_line[row] == [0, 255, 0], axis=1))))

cv2_imshow(img_line)

**Pengamatan Langkah 4.** Baris tengah citra `fruits.jpg` (baris ke-240 dari 480 baris) berubah
menjadi hijau murni, sedangkan baris lain tidak tersentuh karena penugasan hanya mengenai satu baris.
Jumlah pixel yang berubah tepat sama dengan lebar citra (512 pixel). Karena citra BGR, `[0, 255, 0]`
berarti B=0, G=255, R=0 -> hijau; untuk warna merah nilai yang benar adalah `[0, 0, 255]`.

> Alternatif yang lebih cepat adalah fungsi bawaan `cv.line(img, (x1, y1), (x2, y2), warna, tebal)`.
> Pada praktikum ini penulisan garis sengaja dilakukan manual dengan penugasan pixel agar konsep
> "mengakses dan mengubah pixel" terlihat jelas; pada bagian tugas, hasil keduanya dibandingkan.

---

## Jawaban Pertanyaan Praktikum D2

### 1. Mengapa eksekusi kode Python pada modul praktikum ini dilakukan menggunakan Google Colab?

Karena jobsheet ini dirancang untuk praktikum yang harus bisa dijalankan semua mahasiswa tanpa
urusan instalasi dan tanpa spesifikasi perangkat keras tertentu:

* **Berbasis cloud dan gratis** — Colab adalah notebook Jupyter yang berjalan di server Google,
  sehingga Python beserta pustaka yang dibutuhkan (NumPy, pandas, OpenCV, scikit-image, PIL,
  matplotlib) sudah tersedia dan siap dipakai tanpa `pip install` di laptop masing-masing.
* **Akses GPU/TPU gratis** — untuk materi lanjutan (deep learning, YOLO, SAM 2) Colab menyediakan
  akselerator yang tidak dimiliki laptop biasa; Colab Pro++ bahkan memberi A100/H100.
* **Integrasi penyimpanan & version control** — dataset dan hasil eksperimen dapat disimpan di
  Google Drive, dan notebook bisa langsung dihubungkan/disimpan ke repository GitHub (bagian D1).
* **Kolaborasi dan portabilitas** — satu tautan notebook dapat dibuka, dijalankan, dan diedit
  bersama; tidak bergantung pada OS (Windows/Linux/macOS) maupun IDE lokal.
* **Umpan balik langsung** — sel kode dan sel teks bercampur dalam satu dokumen, sehingga kode
  pengolahan citra dapat langsung dilihat hasilnya (termasuk `cv2_imshow` yang menampilkan citra
  tanpa jendela GUI).

Pada versi rework ini alasan yang sama dipenuhi oleh **Kaggle Notebook**: juga notebook Jupyter
berbasis cloud dan gratis, sudah memuat NumPy/OpenCV/scikit-image/pandas/matplotlib, serta
menyediakan dataset yang otomatis ter-mount di `/kaggle/input` sehingga tidak perlu mengunggah file
atau me-mount Google Drive. Bedanya, Kaggle tidak memiliki `google.colab.patches.cv2_imshow` dan
tidak ada GUI, sehingga pada sel konfigurasi dibuat shim `cv2_imshow` berbasis matplotlib. Karena
sel konfigurasi juga mendeteksi folder `datasets/` secara otomatis, notebook yang sama tetap dapat
dijalankan di laptop (VS Code/Jupyter lokal) — bukti bahwa kode yang ditulis memang portabel.

### 2. Kegunaan setiap library pada praktikum langkah ke-8 (sel import). Apakah semua harus dipakai?

| Library | Kegunaan pada praktikum ini |
|---|---|
| **NumPy** (`import numpy as np`) | fondasi representasi citra: citra adalah `ndarray`. Dipakai untuk atribut array (`shape`, `size`, `dtype`, `nbytes`), operasi vektor/matriks, dan pengecekan pixel (mis. `np.all`, `np.array_equal`). |
| **pandas** (`import pandas as pd`) | pengolahan data tabular (DataFrame) — bukan untuk memproses citra, melainkan untuk merekap hasil percobaan (pada notebook ini dipakai menyusun tabel ringkasan ukuran citra pada Tugas 1). |
| **OpenCV** (`import cv2 as cv`) | library utama: membaca/menulis citra (`imread`/`imwrite`), resize, konversi ruang warna (`cvtColor`), pemisahan kanal (`split`), penggabungan citra (`hconcat`), dan fungsi pengolahan citra lain. Bekerja dengan urutan kanal **BGR**. |
| **scikit-image** (`from skimage import io`) | pemrosesan citra tingkat lanjut (filtering, edge detection, morfologi) dan I/O citra. Pada praktikum ini dipakai `io.imread` sebagai pembanding cara membaca citra (menghasilkan **RGB**). |
| **PIL/Pillow** (`from PIL import Image`) | membuka/menyimpan citra, konversi format, resize/rotate/crop dengan API objek `Image`. Di sini dipakai sebagai pembanding ketiga cara membaca citra (`Image.open` -> **RGB**). |
| **matplotlib** (`import matplotlib.pylab as plt`) | visualisasi: menampilkan citra (`imshow`) beserta judul/panel, dan menggambar grafik. Karena mengasumsikan **RGB**, array BGR dari OpenCV harus dikonversi lebih dulu. |

**Apakah semuanya harus digunakan? Tidak.** Library yang mutlak untuk alur praktikum ini adalah
**NumPy** (struktur data citra), **OpenCV** (membaca, resize, konversi, menggabungkan), dan
**matplotlib** (menampilkan hasil). **scikit-image** dan **PIL** bersifat alternatif/komplemen —
keduanya bisa menggantikan fungsi baca-tampil milik OpenCV, dan pada jobsheet dipakai justru untuk
memperlihatkan bahwa urutan kanal tiap library berbeda. **pandas** sama sekali tidak diperlukan
untuk operasi citra; ia hanya berguna jika hasil percobaan perlu direkap dalam bentuk tabel. Jadi
importnya boleh dilakukan agar daftar pustaka lengkap, tetapi pemakaiannya bersifat opsional.

### 3. Kegunaan potongan kode pada uji coba langkah ke-9, dan pengaruhnya bila tidak dijalankan

Potongan kode pada langkah tersebut adalah baris **konversi ruang warna** yang memindahkan citra dari
urutan kanal BGR ke RGB (pada notebook ini: `img = cv.cvtColor(img, cv.COLOR_BGR2RGB)`).

* **Kegunaannya:** `cv.imread` mengembalikan array dengan urutan kanal **BGR**, sedangkan matplotlib,
  PIL, dan skimage mengasumsikan **RGB**. `cv.cvtColor(..., cv.COLOR_BGR2RGB)` menukar kanal 0 dengan
  kanal 2 sehingga citra tampil dengan warna yang benar ketika digambar oleh matplotlib.
* **Pengaruhnya jika tidak dilakukan:** data pixel tidak berubah bentuk maupun ukurannya, tetapi
  kanal merah dan biru menjadi tertukar ketika divisualisasikan dengan library berorientasi RGB.
  Akibatnya warna merah tampak biru dan sebaliknya — pada `fruits.jpg`, buah stroberi yang semula
  merah akan terlihat biru dan bagian langit yang biru akan terlihat kemerahan; pada
  `starry_night.jpg` langit biru gelap bergeser ke warna kekuningan/merah. Bukti visualnya adalah
  keluaran `cv2_imshow(img_merge)` pada Langkah 2: separuh kiri (BGR asli, ditafsirkan sebagai BGR)
  benar, sedangkan separuh kanan (sudah RGB tetapi ditafsirkan sebagai BGR) tampak tertukar warnanya.
  Jadi kesalahan ini tidak menimbulkan *error* program, melainkan kesalahan persepsi warna — jenis
  bug yang paling berbahaya karena tidak terdeteksi tanpa inspeksi visual.

> Catatan: operasi lain pada langkah yang sama adalah `cv.hconcat([...])`, yang menggabungkan citra
> asli dan citra hasil konversi secara horizontal agar keduanya dapat dibandingkan dalam satu
> tampilan. Tanpa operasi itu, kedua citra hanya bisa ditampilkan terpisah dan perbedaan urutan
> kanal tidak langsung terlihat berdampingan.

### 4. Apakah kegunaan `[0, 255, 0]`? Jelaskan!

`[0, 255, 0]` adalah **triplet nilai intensitas tiga kanal** yang ditugaskan ke sebuah pixel, mis.
`img[row, col] = [0, 255, 0]`. Nilai tersebut dibaca dalam urutan **BGR** (urutan kanal OpenCV),
sehingga artinya:

* kanal **B** (Blue) = 0 -> kanal biru mati,
* kanal **G** (Green) = 255 -> kanal hijau maksimum (8 bit = `1111 1111`),
* kanal **R** (Red) = 0 -> kanal merah mati.

Kombinasi tersebut menghasilkan **warna hijau murni**. Karena itu, perulangan
`for col in range(w): img_line[row, col] = [0, 255, 0]` melukis garis horizontal hijau penuh pada
satu baris citra. Dua hal penting yang ditunjukkan kode ini:

1. Nilai `255` adalah intensitas maksimum untuk citra 8 bit (`uint8`), sedangkan `0` berarti kanal
   tersebut tidak berkontribusi; semua warna lain (mis. kuning `[0, 255, 255]`, merah `[0, 0, 255]`,
   putih `[255, 255, 255]`) adalah campuran ketiga kanal.
2. Urutan kanal bersifat mutlak: karena OpenCV memakai BGR, menuliskan `[0, 255, 0]` tetap hijau
   (kanal tengah), tetapi bila ingin merah harus ditulis `[0, 0, 255]`, dan bila kode yang sama
   dijalankan pada library berorientasi RGB (mis. PIL), penafsirannya berbeda. Kesalahan urutan
   kanal menghasilkan warna yang salah tanpa pesan error.

### 5. Keterkaitan antara pixel dan resolusi gambar yang tinggi maupun rendah

**Pixel** adalah elemen terkecil pembentuk citra; setiap pixel menyimpan satu nilai intensitas
(citra grayscale) atau satu triplet nilai kanal (citra RGB) yang menentukan warna pada satu titik.
**Resolusi** adalah banyaknya pixel yang menyusun citra, dinyatakan sebagai lebar x tinggi, sehingga
resolusi = jumlah pixel total. Keduanya terhubung langsung: resolusi adalah "hitungan" dari pixel,
dan kualitas detail citra ditentukan oleh seberapa banyak pixel yang mewakili satu objek.

Keterkaitan tersebut terlihat konkret pada data praktikum ini:

| Citra | Resolusi | Jumlah pixel | Data mentah (nbytes) |
|---|---|---|---|
| `fruits.jpg` | 512 x 480 | 245.760 | 737.280 byte (~0,70 MB) |
| `butterfly.jpg` | 493 x 356 | 175.508 | 526.524 byte (~0,50 MB) |
| `starry_night.jpg` | 752 x 600 | 451.200 | 1.353.600 byte (~1,29 MB) |

Pola yang tampak: `nbytes = tinggi x lebar x jumlah kanal x itemsize`, jadi **resolusi tinggi berarti
lebih banyak pixel, lebih banyak memori, dan waktu komputasi lebih besar**. Sebaliknya, pada citra
beresolusi rendah satu pixel harus mewakili area yang lebih luas, sehingga detail halus hilang dan
tepi objek tampak bergerigi (*jagged/aliasing*) — inilah alasan operasi pada Langkah 2 (`cv.resize`
dengan `fx = fy = 0.5`) memangkas jumlah pixel menjadi seperempatnya: `fruits.jpg` dari 245.760
menjadi 61.440 pixel, sehingga pemrosesan lebih cepat tetapi detailnya berkurang.

Dengan demikian, keputusan resolusi selalu berupa kompromi antara **kualitas visual/detail** dan
**biaya** (ukuran penyimpanan, bandwidth, waktu proses). Citra medis atau penginderaan jauh menuntut
resolusi tinggi karena detail kecil bersifat kritis, sedangkan thumbnail dan aplikasi real-time
memilih resolusi rendah demi kecepatan. Kompromi ini juga menjelaskan mengapa teknik
*super-resolution* berbasis AI (mis. ESRGAN/Real-ESRGAN) bernilai tinggi: ia berusaha menambah detail
pada citra beresolusi rendah tanpa mengambil ulang citra aslinya.

---

## Tugas Praktikum D2

### Tugas 1 — Lakukan langkah-langkah praktikum seperti di atas

Seluruh langkah D2 sudah dikerjakan pada bagian sebelumnya (Langkah 1–4). Sebagai rekap, alur lengkap
dibungkus menjadi satu fungsi `pipeline()` dan dijalankan untuk ketiga citra, lalu hasilnya
direkap dalam tabel memakai pandas.

In [ ]:
# Tugas 1 - rekap alur D2: baca -> resize 1/2 -> konversi RGB -> gabung -> ukuran
def pipeline(path, fx=0.5, fy=0.5):
    img    = cv.imread(path)                                  # 1. baca citra (BGR)
    half   = cv.resize(img, (0, 0), fx=fx, fy=fy)             # 2. resize setengah
    rgb    = cv.cvtColor(half, cv.COLOR_BGR2RGB)              # 3. konversi BGR -> RGB
    merged = cv.hconcat([half, rgb])                          # 4. gabungkan asli + konversi
    return img, half, rgb, merged

rows = []
for name, path in zip(imagenames, imgpaths):
    img, half, rgb, merged = pipeline(path)
    rows.append({
        'nama citra'    : name,
        'shape asli'    : str(img.shape),
        'shape 1/2'     : str(half.shape),
        'pixel asli'    : img.shape[0] * img.shape[1],
        'pixel 1/2'     : half.shape[0] * half.shape[1],
        'nbytes asli'   : img.nbytes,
        'nbytes 1/2'    : half.nbytes,
    })
    print('{} -> citra gabungan {}'.format(name, merged.shape))
    cv2_imshow(merged)

try:
    from IPython.display import display
except Exception:
    display = print

df = pd.DataFrame(rows)
df['rasio pixel'] = (df['pixel asli'] / df['pixel 1/2']).round(2)
display(df)
print('Total pixel seluruh citra (1/2 ukuran):', int(df['pixel 1/2'].sum()))

**Analisis Tugas 1.** Ketiga citra berhasil melewati alur lengkap tanpa error. `cv.resize` dengan
`fx = fy = 0.5` membuat tinggi dan lebar menjadi setengah, sehingga jumlah pixel turun menjadi
**seperempat** (kolom `rasio pixel` menunjukkan 4,00 untuk `fruits.jpg` dan `starry_night.jpg`,
serta 4,01 untuk `butterfly.jpg` karena lebarnya ganjil — 493 -> 246 — sehingga pembulatan ke bawah
membuat rasio sedikit di atas 4). `nbytes` juga turun menjadi seperempat: `fruits.jpg` dari 737.280
byte menjadi 184.320 byte. Penggabungan `cv.hconcat` menghasilkan citra dengan lebar dua kali lipat
(`512 -> 1024` untuk fruits, `752 -> 1504` untuk starry_night), karena `hconcat` mensyaratkan kedua
bagian memiliki tinggi yang sama — dan tinggi `img_half` dengan `img_rgb` memang sama, sebab
`cvtColor` hanya mengubah urutan kanal, tidak mengubah dimensi citra. Tampilan `cv2_imshow`
memperlihatkan separuh kiri benar dan separuh kanan tertukar kanal merah/biru — bukti bahwa konversi
`COLOR_BGR2RGB` benar-benar bekerja. Tabel rekap ini juga contoh pemakaian **pandas** yang disebut
pada pertanyaan nomor 2: library tersebut tidak memproses citra, hanya menata hasil percobaan.

### Tugas 2 — Garis vertikal dan garis menyilang (diagonal) pada citra keluaran

Dua garis diagonal dibuat bersilangan membentuk huruf "X" (diagonal utama dan diagonal
anti-utama), ditambah satu garis vertikal di tengah citra. Semua garis digambar **manual dengan
penugasan pixel**, lalu hasilnya diverifikasi terhadap fungsi bawaan `cv.line`.

In [ ]:
# Tugas 2 - garis vertikal + garis menyilang diagonal (X) secara manual
img = cv.imread(imgpaths[2])            # starry_night.jpg, shape (600, 752, 3)
h, w = img.shape[:2]
HIJAU = [0, 255, 0]
MERAH = [0, 0, 255]
col = w // 2                            # kolom tengah
d   = min(h, w)                         # panjang diagonal maksimum

# (a) hanya garis vertikal di tengah
vert = img.copy()
for row in range(h):
    vert[row, col] = HIJAU

# (b) garis vertikal + garis menyilang (diagonal utama & anti-diagonal)
out = vert.copy()
for i in range(d):
    out[i, i]         = MERAH           # diagonal utama: kiri-atas -> kanan-bawah
    out[i, w - 1 - i] = MERAH           # anti-diagonal: kiri-bawah -> kanan-atas

print('citra            :', img.shape)
print('garis vertikal   : kolom', col, 'sepanjang', h, 'pixel')
print('diagonal utama   : {} pixel, berakhir di (baris, kolom) = {}'.format(d, (d - 1, d - 1)))
print('anti-diagonal    : {} pixel, berakhir di (baris, kolom) = {}'.format(d, (d - 1, w - d)))
print('sudut kanan-bawah citra =', (h - 1, w - 1), '(tidak tersentuh karena lebar > tinggi)')

# verifikasi penulisan manual vs fungsi bawaan cv.line, pada citra yang hanya bergaris vertikal
ref = img.copy()
cv.line(ref, (col, 0), (col, h - 1), (0, 255, 0), 1)
print()
print('garis vertikal manual == cv.line :', np.array_equal(vert, ref))
print('jumlah pixel hijau pada kolom {}:'.format(col), int(np.sum(np.all(vert[:, col] == HIJAU, axis=1))))

# titik potong pada citra gabungan (diagonal digambar setelah garis vertikal)
print()
print('pixel pada daerah persilangan -> (B, G, R):')
for r, c in [(375, 375), (375, 376), (376, 375), (376, 376), (300, 376)]:
    print('   out[{}, {}] = {}'.format(r, c, out[r, c]))
print('jumlah pixel merah pada citra gabungan:', int(np.sum(np.all(out == MERAH, axis=2))))
print('jumlah pixel hijau pada citra gabungan:', int(np.sum(np.all(out == HIJAU, axis=2))),
      '(= {} baris dikurangi 2 pixel yang tertimpa diagonal)'.format(h))

cv2_imshow(out)

**Analisis Tugas 2.** Garis vertikal hijau menempati kolom ke-376 (tengah dari lebar 752) dan
menyentuh seluruh 600 baris citra; hasil penulisan manual ini identik dengan `cv.line` (perbandingan
`np.array_equal` bernilai `True`), sehingga cara mengakses dan mengubah pixel sudah tepat. Dua garis
diagonal merah digambar sepanjang `d = min(tinggi, lebar) = 600` pixel: diagonal utama dari (0,0)
sampai (599,599) dan anti-diagonal dari (0,751) sampai (599,152). Karena `starry_night.jpg` tidak
persegi (752 x 600), kedua diagonal berhenti di baris terakhir dan **tidak menyentuh sudut
kanan-bawah** (599,751) maupun sudut kiri-bawah (599,0) — garisnya terpotong lebih dulu oleh tepi
atas/bawah citra, bukan oleh sisi kiri/kanan. Kedua diagonal bersilangan di sekitar baris 375-376
dan kolom 375-376, yaitu titik di mana `i` mendekati `w - 1 - i`. Pada daerah persilangan itu, dua
pixel di kolom 376 (baris 375 dan 376) berubah dari hijau menjadi merah karena loop diagonal
dijalankan **setelah** loop garis vertikal: urutan penggambaran menentukan warna akhir pada pixel
yang saling menimpa, sehingga jumlah pixel hijau menjadi 598, bukan 600. Titik tengah citra
`out[300, 376]` tetap hijau karena hanya dilalui garis vertikal, sedangkan titik potong kedua
diagonal berada di baris 375-376, bukan di baris 300 — akibat lain dari citra yang tidak persegi.
Hasil manual dan hasil library (`cv.line`) sama; perbedaannya hanya pada efisiensi, karena
perulangan Python memproses satu pixel per iterasi sedangkan fungsi bawaan OpenCV menjalankan
algoritma Bresenham di level C.

### Tugas 3 — Garis horizontal berwarna merah di bagian tengah gambar dengan panjang tertentu

Garis dibuat berpusat pada titik tengah citra dengan panjang `L` pixel (di sini 350 pixel, sekitar
68% lebar citra), sehingga titik awalnya adalah `(w - L) // 2`.

In [ ]:
# Tugas 3 - garis horizontal merah di tengah citra dengan panjang tertentu
img = cv.imread(imgpaths[0])            # fruits.jpg, shape (480, 512, 3)
out = img.copy()
h, w = out.shape[:2]

L = 350                                 # panjang garis yang diinginkan (pixel)
L = min(L, w)                           # jaga agar tidak melebihi lebar citra
row  = h // 2                           # baris tengah
c0   = (w - L) // 2                     # kolom awal agar garis terpusat
c1   = c0 + L                           # kolom akhir (eksklusif)

for c in range(c0, c1):
    out[row, c] = [0, 0, 255]           # BGR: B=0, G=0, R=255 -> merah murni

print('ukuran citra        :', out.shape, '-> baris tengah', row, ', lebar', w)
print('panjang garis       :', L, 'pixel')
print('kolom awal - akhir  :', c0, '-', c1 - 1)
print('simetri thd tengah  : c0 == w - c1 -> {} ({} == {})'.format(c0 == w - c1, c0, w - c1))
print('jarak ke tepi kiri  : {} pixel | ke tepi kanan: {} pixel'.format(c0, w - c1))
print('pixel [{},{}]        :'.format(row, c0), out[row, c0], '(B,G,R)')
print('pixel sebelum garis :', out[row, c0 - 1], '| pixel sesudah garis:', out[row, c1])
print('jumlah pixel merah  :', int(np.sum(np.all(out[row] == [0, 0, 255], axis=1))))

cv2_imshow(out)

**Analisis Tugas 3.** Garis merah menempati baris ke-240 (tepat di tengah tinggi 480) dan membentang
dari kolom 81 sampai 430, yaitu tepat 350 pixel — sama dengan nilai `L` yang diminta, dibuktikan oleh
penghitungan `np.sum(np.all(out[row] == [0, 0, 255], axis=1))` yang menghasilkan 350. Garis
benar-benar terpusat karena `c0 = (w - L) // 2 = 81` dan jarak ke tepi kanan juga 81 pixel
(`c0 == w - c1` bernilai `True`), sehingga garis simetris terhadap titik tengah citra pada arah
horizontal. Nilai `[0, 0, 255]` adalah BGR untuk **merah murni**: kanal biru dan hijau nol, kanal
merah maksimum 255 — kebalikan dari `[0, 255, 0]` (hijau) pada Langkah 4, dan contoh konkret bahwa
pada OpenCV urutan penulisan komponen warna adalah B-G-R. Pixel tepat di luar ujung garis
(`out[240, 80]` = [27, 139, 182] dan `out[240, 431]` = [41, 84, 69]) masih bernilai asli, yang
membuktikan panjang garis terkendali tepat sesuai rentang kolom yang diiterasi.

### Tugas 4 — Kotak dari kumpulan pixel berwarna putih di sembarang tempat

Kotak dibuat dari **kumpulan pixel putih** `[255, 255, 255]` pada daerah persegi berukuran
`size x size` dengan sudut kiri-atas di `(r0, c0)`, memakai perulangan bersarang (baris dan kolom).
Pada notebook ini kotak diarsir penuh (*filled box*) agar jelas terlihat berasal dari pixel; variasi
kerangka (*outline*) juga ditampilkan sebagai pembanding.

In [ ]:
# Tugas 4 - kotak dari kumpulan pixel putih di sembarang tempat
img = cv.imread(imgpaths[1])            # butterfly.jpg, shape (356, 493, 3)
out = img.copy()
h, w = out.shape[:2]

r0, c0, size = 60, 320, 120             # sudut kiri-atas dan ukuran kotak (pixel)
r1, c1 = min(r0 + size, h), min(c0 + size, w)

# (a) kotak penuh: seluruh pixel pada daerah kotak diwarnai putih
for r in range(r0, r1):
    for c in range(c0, c1):
        out[r, c] = [255, 255, 255]

print('ukuran citra       :', out.shape)
print('daerah kotak       : baris {}..{} , kolom {}..{}'.format(r0, r1 - 1, c0, c1 - 1))
print('ukuran kotak       : {} x {} pixel'.format(r1 - r0, c1 - c0))
putih_asli = int(np.sum(np.all(img == [255, 255, 255], axis=2)))
putih_kotak = int(np.sum(np.all(out[r0:r1, c0:c1] == [255, 255, 255], axis=2)))
print('pixel putih pada citra asli        :', putih_asli, '(bercak putih pada sayap kupu-kupu)')
print('pixel putih di dalam daerah kotak  :', putih_kotak, '= {} x {}'.format(r1 - r0, c1 - c0))
print('pixel dalam kotak  :', out[r0, c0], '| pixel di luar:', out[0, 0])

# (b) pembanding: kerangka kotak (outline) dari pixel putih juga
outline = img.copy()
for c in range(c0, c1):                 # sisi atas & bawah
    outline[r0, c] = [255, 255, 255]
    outline[r1 - 1, c] = [255, 255, 255]
for r in range(r0, r1):                 # sisi kiri & kanan
    outline[r, c0] = [255, 255, 255]
    outline[r, c1 - 1] = [255, 255, 255]

putih_outline = int(np.sum(np.all(outline == [255, 255, 255], axis=2))) - putih_asli
print('pixel putih pada kerangka kotak    :', putih_outline,
      '= 2 x {} + 2 x {} - 4 sudut'.format(r1 - r0, c1 - c0))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(cv.cvtColor(out, cv.COLOR_BGR2RGB));     ax[0].set_title('kotak penuh (filled)'); ax[0].axis('off')
ax[1].imshow(cv.cvtColor(outline, cv.COLOR_BGR2RGB)); ax[1].set_title('kerangka kotak (outline)'); ax[1].axis('off')
plt.tight_layout(); plt.show()

cv2_imshow(out)

**Analisis Tugas 4.** Kotak putih berukuran 120 x 120 pixel menempati daerah baris 60-179 dan kolom
320-439 pada `butterfly.jpg`, yaitu di kuadran kanan-atas citra (sembarang tempat, sesuai permintaan
tugas). Jumlah pixel di dalam daerah kotak yang bernilai `[255, 255, 255]` tepat **14.400**
(= 120 x 120), yang berarti seluruh pixel daerah itu benar-benar tergantikan oleh putih. Perlu
dicatat bahwa `butterfly.jpg` sudah memiliki pixel putih pada sayap kupu-kupu, sehingga menghitung
putih di seluruh citra memberi angka lebih besar daripada luas kotak; karena itu penghitungan
dilakukan pada irisan `out[60:180, 320:440]` saja. Nilai `[255, 255, 255]` adalah intensitas
maksimum pada ketiga kanal, jadi hasilnya putih rata baik dibaca sebagai BGR maupun RGB (urutan
kanal tidak berpengaruh untuk putih). Versi kerangka (*outline*) memakai pixel putih hanya pada
empat sisinya: 2 x 120 pixel untuk sisi atas/bawah ditambah 2 x 120 pixel untuk sisi kiri/kanan
dikurangi 4 pixel sudut yang terhitung dua kali, sehingga jumlahnya 476 pixel. Baik versi penuh
maupun kerangka dapat digantikan oleh satu pemanggilan `cv.rectangle`, tetapi penulisan manual di
sini memperlihatkan mekanisme indexing dua dimensi `img[baris, kolom]` yang menjadi inti praktikum
D2.

In [ ]:
# Menyimpan seluruh citra keluaran praktikum & tugas
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs(OUT, exist_ok=True)

# Langkah 4: garis hijau
img = cv.imread(imgpaths[0]); g = img.copy()
g[g.shape[0] // 2, :] = [0, 255, 0]
cv.imwrite(os.path.join(OUT, 'week1_langkah4_garis_hijau.jpg'), g)

# Tugas 2: garis vertikal + diagonal
img = cv.imread(imgpaths[2]); t2 = img.copy()
hh, ww = t2.shape[:2]
for r in range(hh):
    t2[r, ww // 2] = [0, 255, 0]
for i in range(min(hh, ww)):
    t2[i, i] = [0, 0, 255]
    t2[i, ww - 1 - i] = [0, 0, 255]
cv.imwrite(os.path.join(OUT, 'week1_tugas2_vertikal_diagonal.jpg'), t2)

# Tugas 3: garis merah terpusat
img = cv.imread(imgpaths[0]); t3 = img.copy()
rr, w2 = t3.shape[:2]
for c in range((w2 - 350) // 2, (w2 - 350) // 2 + 350):
    t3[rr // 2, c] = [0, 0, 255]
cv.imwrite(os.path.join(OUT, 'week1_tugas3_garis_merah.jpg'), t3)

# Tugas 4: kotak putih
img = cv.imread(imgpaths[1]); t4 = img.copy()
for r in range(60, 180):
    for c in range(320, 440):
        t4[r, c] = [255, 255, 255]
cv.imwrite(os.path.join(OUT, 'week1_tugas4_kotak_putih.jpg'), t4)

print('Citra keluaran tersimpan di:', os.path.abspath(OUT))
for f in sorted(os.listdir(OUT)):
    if f.startswith('week1_'):
        print('  ', f, '->', os.path.getsize(os.path.join(OUT, f)), 'byte')

---

## Kesimpulan

1. **Citra digital = ndarray.** Citra berwarna hasil `cv.imread` adalah array `uint8` ber-rank 3
   dengan `shape` (tinggi, lebar, kanal); seluruh atribut NumPy (`ndim`, `shape`, `size`, `dtype`,
   `itemsize`, `nbytes`) langsung berlaku dan menjadi dasar perhitungan ukuran citra.
2. **Urutan kanal adalah sumber bug tersembunyi.** OpenCV memakai BGR, sedangkan matplotlib, PIL, dan
   skimage memakai RGB. Tanpa `cv.cvtColor(..., cv.COLOR_BGR2RGB)` program tetap berjalan tanpa
   error, tetapi warna merah dan biru tertukar saat ditampilkan.
3. **Beberapa library menawarkan fungsi yang tumpang tindih.** `cv.imread`, `io.imread`,
   `Image.open`, dan `plt.imread` sama-sama bisa membaca citra dengan urutan kanal dan tipe data yang
   berbeda-beda; pandas tidak diperlukan untuk operasi citra dan hanya berguna untuk merekap hasil.
4. **Resolusi menentukan biaya dan detail.** Jumlah pixel = `tinggi x lebar`; menurunkan ukuran
   menjadi setengah (faktor 0,5 pada `cv.resize`) memangkas jumlah pixel menjadi seperempat sehingga
   proses lebih cepat, dengan konsekuensi detail berkurang.
5. **Pixel dapat diakses dan diubah langsung** melalui indexing `img[baris, kolom]`, sehingga garis
   horizontal/vertikal/diagonal maupun kotak dapat dibentuk dari kumpulan pixel. Hasil penulisan
   manual terbukti identik dengan fungsi bawaan `cv.line`, perbedaannya hanya pada efisiensi.

Seluruh materi jobsheet Minggu 1 — termasuk praktikum D1, praktikum D2, pertanyaan, dan tugas —
telah dijalankan pada notebook ini dan seluruh sel berjalan tanpa error.